# DisasterIQ — Kaggle GPU fine-tune

Fine-tune PyTorch xView2 on your pre-built **train_subset** (~1449 pairs).

**Before running:**
1. Upload `disasteriq-train-subset.zip` as a private Kaggle dataset (see `docs/KAGGLE_FINETUNE.md`)
2. **Input** → add dataset `disasteriq-train-subset` (optional: tier3 for more data)
3. **Settings** → **Accelerator** → **GPU T4 x2** (recommended; P100 needs cell 2 cu118 PyTorch)

### Critical — avoid "Draft Session off" wiping checkpoints
Interactive cell-by-cell runs die when the browser sleeps / websocket drops, and `/kaggle/working` can wipe **before any epoch-end checkpoint**.

**For real training:** use **Save Version → Save & Run All (Commit)**. That runs on Kaggle’s servers without needing your laptop open. Then download `damage_best.ckpt` from the completed version’s **Output**.

Training now checkpoints mid-epoch (`val_check_interval=0.25` + step every 400 batches) via `overrides/main.py`.

**Interactive smoke order (optional):** 1 → 2 → 3 → 4 (CPU) → enable GPU → 5 → then Commit for cell 6.

**Resume (localization done, damage failed):** run cell 6b — do **not** use `git pull` (local script edits block it on Kaggle).

In [ ]:
DATASET_SLUG = "disasteriq-train-subset"
REPO_URL = "https://github.com/AhmadRaza4076/DisasterIQ.git"
REPO_BRANCH = "main"
XVIEW2_URL = "https://github.com/michal2409/xView2.git"

from pathlib import Path

WORKING = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO_ROOT = WORKING / "DisasterIQ"


def find_train_subset(root: Path) -> Path | None:
    if (root / "images").is_dir() and (root / "targets").is_dir():
        return root
    for child in sorted(root.iterdir()):
        if child.is_dir():
            found = find_train_subset(child)
            if found:
                return found
    return None


dataset_path = find_train_subset(INPUT)
if dataset_path is None:
    print("Available input dirs:", [p.name for p in INPUT.iterdir()] if INPUT.exists() else [])
    raise FileNotFoundError(
        f"Dataset not found. Add Input dataset '{DATASET_SLUG}' with images/ labels/ targets/"
    )
print("Dataset OK:", dataset_path)
print("Images:", len(list((dataset_path / "images").glob("*"))))

In [ ]:
# Default Kaggle PyTorch may not support P100 (sm_60). cu118 wheels work on P100 and T4.
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
# Pinned deps for xView2 on Kaggle (see ml/finetune/requirements_kaggle.txt after clone)
!pip install -q "pytorch-lightning==1.9.5" torchmetrics torch-optimizer timm segmentation-models-pytorch
!pip install -q albumentations opencv-python-headless pyyaml joblib tqdm shapely fire "monai>=1.3,<2"
# NVIDIA DLLogger is NOT the PyPI 'dllogger' package — must install from GitHub
!pip install -q "git+https://github.com/NVIDIA/dllogger.git#egg=dllogger"

In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/AhmadRaza4076/DisasterIQ.git"
REPO_BRANCH = "main"
XVIEW2_URL = "https://github.com/michal2409/xView2.git"
WORKING = Path("/kaggle/working")
REPO_ROOT = WORKING / "DisasterIQ"

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
os.chdir(WORKING)
!git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} DisasterIQ

xview2_dir = REPO_ROOT / "ml" / "pytorch-xview2"
if xview2_dir.exists():
    shutil.rmtree(xview2_dir)
!git clone --depth 1 {XVIEW2_URL} {xview2_dir}
print("Repos ready at", REPO_ROOT)

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/DisasterIQ")
if not REPO_ROOT.is_dir():
    raise RuntimeError("DisasterIQ repo missing — run cell 3 (clone) first.")

os.chdir(REPO_ROOT)
!chmod +x ml/finetune/run_kaggle_pipeline.sh ml/finetune/train_localization.sh ml/finetune/train_damage.sh
!bash ml/finetune/run_kaggle_pipeline.sh --prep-only

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. Settings → Accelerator → GPU T4 x2 (or P100) → Save, restart, re-run."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
# Fail fast if PyTorch build does not support this GPU (e.g. P100 sm_60 on default torch)
_ = torch.randn(4, 4, device="cuda")
print("CUDA tensor test: OK")

In [ ]:
# Full training: prep patches + localization (5 ep) + damage (8 ep)
import os
import subprocess
import sys
import urllib.request
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/DisasterIQ")
FINETUNE = REPO_ROOT / "ml/finetune"
os.chdir(REPO_ROOT)

# Fetch latest scripts from GitHub (avoids git pull conflicts from prior Kaggle edits)
RAW = "https://raw.githubusercontent.com/AhmadRaza4076/DisasterIQ/main/ml/finetune"
FETCH = (
    "kaggle_train.py",
    "patch_pytorch_xview2.py",
    "smoke_damage_step.py",
    "fetch_kaggle_scripts.py",
    "load_config.py",
    "config_subset_kaggle.yaml",
    "overrides/main.py",
    "overrides/model/loss.py",
)
for name in FETCH:
    dest = FINETUNE / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(urllib.request.urlopen(f"{RAW}/{name}", timeout=60).read())
    print("Fetched", name)

subprocess.run(
    [sys.executable, "ml/finetune/kaggle_train.py", "--stage", "all"],
    check=True,
)

In [ ]:
# Cell 6b — Resume DAMAGE only (localization already done). Run after cell 5 (GPU check).
import os
import subprocess
import sys
import urllib.request
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/DisasterIQ")
FINETUNE = REPO_ROOT / "ml/finetune"
os.chdir(REPO_ROOT)

RAW = "https://raw.githubusercontent.com/AhmadRaza4076/DisasterIQ/main/ml/finetune"
FETCH = (
    "kaggle_train.py",
    "patch_pytorch_xview2.py",
    "smoke_damage_step.py",
    "load_config.py",
    "config_subset_kaggle.yaml",
    "overrides/main.py",
    "overrides/model/loss.py",
)
for name in FETCH:
    dest = FINETUNE / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(urllib.request.urlopen(f"{RAW}/{name}", timeout=60).read())
    print("Fetched", name)

subprocess.run(
    [sys.executable, "ml/finetune/kaggle_train.py", "--stage", "dmg", "--skip-deps"],
    check=True,
)
# Smoke test is OFF by default; damage trains directly after patches.

In [ ]:
from pathlib import Path
import shutil

ckpt = Path("/kaggle/working/damage_best.ckpt")
dmg_dir = Path("/kaggle/working/results/dmg/checkpoints")

if not ckpt.exists() and dmg_dir.is_dir():
    for candidate in [
        dmg_dir / "best.ckpt",
        dmg_dir / "last.ckpt",
        *sorted(dmg_dir.glob("*.ckpt"), key=lambda p: p.stat().st_mtime, reverse=True),
    ]:
        if candidate.exists():
            shutil.copy(candidate, ckpt)
            print("Copied from", candidate.name)
            break

if ckpt.exists():
    print(f"SUCCESS: {ckpt} ({ckpt.stat().st_size / 1e6:.1f} MB)")
    print("Download from Output tab → damage_best.ckpt")
    print("Laptop: copy to ml/checkpoints/damage_best.ckpt")
    print(".env: INFERENCE_MODE=pytorch")
else:
    raise FileNotFoundError("No checkpoint — check training logs above")